In [17]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")
if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your HF API key: ")

# Model used for every chat call in this notebook. Set SELF_CONSISTENCY_MODEL to override,
# or swap the name on openrouter.ai/models.
MODEL = os.getenv("SELF_CONSISTENCY_MODEL", "openai/gpt-6-luna")

## Self-Consistency Prompting in Language Models

Self-consistency prompting was introduced in the March 2022 paper ["Self-Consistency Improves Chain of Thought Reasoning in Language Models"](https://arxiv.org/pdf/2203.11171.pdf) by Xuezhi Wang et al.

**What you need:** an OpenRouter API key (for the chat and embedding models) and a Hugging Face token (for the example dataset). The setup cell reads both from your `.env` file, or asks you to type them in. The chat model is set by `MODEL` in that same cell.

### What is self-consistency prompting?

In plain terms: ask the model the same question several times, let it reason in a slightly different way each time, then keep the answer that comes up most often.

It builds on Chain-of-Thought (CoT) prompting, where the model writes out its reasoning step by step before giving an answer. Self-consistency adds one idea: a single reasoning path can go wrong, so we look at several paths and trust the answer they agree on. The paper reports accuracy gains from this on arithmetic and commonsense reasoning tasks.

### Differences from other prompting techniques

- **Standard prompting:** the model gives an answer directly.
- **Chain-of-Thought (CoT):** the model writes out one line of reasoning, then the answer.
- **Self-consistency:** the model writes out several lines of reasoning, and the most consistent answer wins.
- It needs no extra training or human-labelled examples, so it works with models that are already trained.

### Constructing a self-consistency prompt


<figure>
  <img src="https://pbs.twimg.com/media/Ff73Y-RaEAANaqT.jpg" alt="Image Description" style="width:100%">
  <figcaption>The self-consistency method contains three steps: (1) prompt a language model using chain-of-thought (CoT) prompting; (2) replace the “greedy decode” in CoT prompting by sampling from the language model’s decoder to generate a diverse set of reasoning paths; and (3) marginalize out the reasoning paths and aggregate by choosing the most consistent answer in the final answer set.</figcaption>
  <a href="https://twitter.com/denny_zhou/status/1584978661668966401">Image Source</a>
</figure>


The figure's "greedy decode" means always picking the single most likely next word. Sampling picks words at random, weighted by how likely each one is, so each run can take a different path.

1. **Start with CoT**: use chain-of-thought prompting as the base.

2. **Sample diverse paths**: instead of always taking the most likely word, sample. Each run then produces a different reasoning path. "Sample-and-marginalize" means: generate several paths, then ignore the paths themselves and keep only their final answers.

3. **Marginalize for consistency**: count the final answers and pick the most common one.

### Examples in action

- **Chain-of-Thought:** For the question "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?", instead of answering "5" straight away, the model might reason: "There are 3 cars in the parking lot already. 2 more arrive. Now there are 3 + 2 = 5 cars. The answer is 5."

- **Self-Consistency:** Janet's ducks lay 16 eggs a day. She eats 3 for breakfast, bakes muffins with 4, and sells the rest at $2 per egg. How much does she make a day? Three possible reasoning paths (illustrative, written for this lesson rather than copied from a model run):
  1. "She has 16 - 3 - 4 = 9 eggs left. So she makes $2 * 9 = $18 per day."

  2. "This means she sells the remainder for $2 * (16 - 4 - 3) = $18 per day."

  3. "She eats 3 for breakfast, so she has 16 - 3 = 13 left. Then she bakes muffins, so she has 13 - 4 = 9 eggs left. So she has 9 eggs * $2 = $18."

  The three paths word their working differently, but they all reach $18. Self-consistency takes the answer that appears most often, so the result is $18 per day.

### Impact

- It can improve a model's reasoning by looking at more than one path.
- The paper reports gains on arithmetic and commonsense reasoning. Results depend on the model and the task.

## Begin by setting up CoT prompts

We're following the same pattern as the Chain-of-Thought lesson. This notebook has two parts: Part 1 builds a CoT prompt from worked examples, and Part 2 turns it into a self-consistency prompt.

**Part 1: Set up the CoT prompt**

1. Download CoT prompt examples from Hugging Face. We keep the first 10,000 from the training split.

2. Create embeddings through OpenRouter. An embedding is a list of numbers that represents the meaning of a piece of text, so similar examples get similar numbers.

3. Create a prompt template for CoT

4. Create an example selector. It picks examples that are relevant to the question but different from each other.

5. Construct the prompt by filling the template with the chosen examples and the question

**Part 2: Self-consistency**

Part 2 uses one question throughout: nine computers are in a server room, and five more are installed each day from Monday to Thursday. The correct answer is 29.

6. Write a self-consistency prompt template

7. Sample five reasoning paths from the model

8. Extract the final answer from each path

9. Choose the majority answer with a vote in Python

10. Ask the model to do the same vote, and compare the two


In [18]:
from datasets import load_dataset

# The original dataset is a loading script, which datasets 4+ no longer runs.
# Read the Parquet copy that Hugging Face generates for the same data instead.
dataset_stream = load_dataset(
    "parquet",
    data_files="hf://datasets/kaist-ai/CoT-Collection@refs/convert/parquet/en/train/*.parquet",
    split="train",
    streaming=True,
    token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
)

# Take the first 10,000 examples from the training split
dataset = dataset_stream.take(10_000)  # type: ignore

# Convert to a list to materialize the data
dataset = list(dataset)


In [19]:
import pandas as pd

# Convert the list of examples to a pandas DataFrame
dataset_df = pd.DataFrame(dataset) #type: ignore

# Remove the 'task' and 'type' columns if they exist
columns_to_remove = [col for col in ['task', 'type'] if col in dataset_df.columns]
if columns_to_remove:
    dataset_df = dataset_df.drop(columns=columns_to_remove)

dataset_df.head()

selected_examples = dataset_df.to_dict(orient='records')

In [20]:
from langchain_openai import OpenAIEmbeddings

# Embeddings go through OpenRouter. langchain_openrouter only provides chat models.
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
)



# from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# model_name = "BAAI/bge-base-en-v1.5"

# encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

# embeddings = HuggingFaceBgeEmbeddings(
#     model_name=model_name,
#     model_kwargs={'device': 'cuda'},
#     encode_kwargs=encode_kwargs
# )

In [21]:
from langchain_core.example_selectors import MaxMarginalRelevanceExampleSelector
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

prefix = "Consider the following as examples of how to reason:"

examples_template = """Query: {source}

Rationale: {rationale}

Response: {target}
"""

suffix = """Using a similar reasoning approach, answer the users question which is delimited by triple backticks.

User question: ```{input}```

Take a deep breath, break down the user's query step-by-step, and provide a clear chain of thought in your response.

Finish with a final line in the form "Response: <answer>".
"""

examples_prompt = PromptTemplate(
    input_variables=["source", "rationale", "target"],
    template=examples_template
)

In [22]:
example_selector = MaxMarginalRelevanceExampleSelector.from_examples(
    selected_examples,
    embeddings,
    InMemoryVectorStore,
    k=5,
)

mmr_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=examples_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input"]
)

In [23]:
query = """There were nine computers in the server room. Five more computers were installed each day, from
monday to thursday. How many computers are now in the server room?
"""
prompt = mmr_prompt.format(input=query)

In [24]:
print(prompt)

Consider the following as examples of how to reason:

Query: In this task, you need to answer the given multiple-choice question on the gain. Gain is the value by which to multiply the input. Classify your answers into 'a', 'b', 'c', 'd', and 'e'.

Problem: running at the same rate , 8 identical machines can produce 560 paperclips a minute . at this rate , how many paperclips could 18 machines produce in 6 minutes ? 
Options: a ) 1344 , b ) 3360 , c ) 7560 , d ) 50400 , e ) 67200

Rationale: The machines produce 560 paperclips a minute when there are 8 of them. Since each machine produces the same amount, we can say that each one produces 560 / 8 = 70 paperclips a minute.\nThere are 18 machines producing at this rate, so they produce in total 18 * 70 = 1260 paperclips per minute. In 6 minutes, they will have produced 6 * 1260 = 7560 paperclips.

Response: c


Query: In this task, you need to count the number of words in a sentence that contain the given letter

Sentence: 'two computer 

## Self-Consistency Prompt

The template below (`sc_template`) asks the model to read a list of responses and return the one that appears most often.

- The responses go inside `< >`, and the query goes inside triple backticks. These markers help the model tell the parts apart.
- The responses are passed in as a Python list, so they appear in square brackets, like `['29', '29']`.

We use this template later, when the model does its own vote (step 10).


In [25]:
sc_template = """Based on the responses (delimited by < >) to the following query, \
(delimited by triple backticks) return the response that occurs most frequently.

Query: ```{query}```

Responses: <{responses}>
"""

sc_prompt = PromptTemplate.from_template(sc_template)

### Generating multiple responses

We want several alternative responses to the same prompt, so we can count how often each answer appears.

The `temperature` setting controls how random the output is. Self-consistency needs variety, so we use `0.7`. Each response then differs a little from the others.

`ChatOpenRouter` could not return several completions in one request in the version this notebook was written with (`langchain-openrouter` 0.2.9, as pinned in `requirements.txt`). Setting `n` (the number of completions) failed with `Chat.send() got an unexpected keyword argument 'n'`. Newer versions may behave differently, so check before relying on this.

So we send the same prompt five times instead. `llm.generate` makes five separate requests, and each one is an independent sample. Samples are random, so two can still come out word for word the same, as two of the five did in the example run below. Your output will differ from ours.


In [26]:
from langchain_core.messages import HumanMessage
from langchain_openrouter import ChatOpenRouter

# temperature > 0 so that the five reasoning paths differ from each other.
llm = ChatOpenRouter(model=MODEL, temperature=0.7)

# Chat models take messages, so wrap each prompt in a HumanMessage.
# Five identical requests give five independent samples.
generations = llm.generate([[HumanMessage(content=prompt)]] * 5)

In [27]:
# Guard: make sure we got 5 samples, one per request.
all_generations = [gen for group in generations.generations for gen in group]
assert len(all_generations) == 5, f"Expected 5 samples, got {len(all_generations)}"

[gen.message.content for gen in all_generations]


['From Monday to Thursday is 4 days. Five computers were installed each day, so 5 × 4 = 20 computers were added. Then 9 + 20 = 29 computers.\n\nResponse: 29',
 'From Monday through Thursday, five computers were installed on each of 4 days: 5 × 4 = 20. Adding the original 9 gives 9 + 20 = 29.\n\nResponse: 29',
 'Five computers were installed each day for four days (Monday through Thursday): 5 × 4 = 20. Add the original nine: 9 + 20 = 29.\n\nResponse: 29',
 'Five computers were installed each day for four days: 5 × 4 = 20. Adding the original nine gives 9 + 20 = 29.\n\nResponse: 29',
 'Five computers were installed each day for four days, adding 5 × 4 = 20 computers. With the original nine, there are 9 + 20 = 29 computers.\n\nResponse: 29']

In [28]:
import re

# Pull out the text after "Response:". Handles plain and bold markdown, e.g. "**Response:** 29".
response_pattern = re.compile(r"\*{0,2}Response\*{0,2}\s*:\s*\*{0,2}\s*(.+)", re.IGNORECASE)

responses = []
for gen in all_generations:
    match = response_pattern.search(gen.message.content)
    if match:
        # Strip stray markdown and trailing punctuation, e.g. "29." -> "29"
        responses.append(match.group(1).strip(" *."))
    else:
        print("No 'Response:' line found in:", gen.message.content[:80])

In [29]:
responses

['29', '29', '29', '29', '29']

### Choosing the answer: majority vote in Python

Count the extracted answers with `Counter`, from Python's standard library. `most_common(1)` returns the most frequent answer and how many times it appeared. In the example run there are five answers, so the count is out of five.

Python does the counting, so the result is exact and repeatable: the same list always gives the same result. A vote shows what the model said most often, not what is true. If two answers tie, `Counter` returns the one seen first, which is a tie-breaking rule rather than evidence.

The next section asks the model to do the same count, so you can compare the two.


In [30]:
from collections import Counter

vote_counts = Counter(responses)
majority_answer, majority_count = vote_counts.most_common(1)[0]

print("Votes:", dict(vote_counts))
print(f"Majority answer: {majority_answer} ({majority_count} of {len(responses)} responses)")

Votes: {'29': 5}
Majority answer: 29 (5 of 5 responses)


### Alternative: ask the model to pick the majority answer

The model can also do the count, using the self-consistency prompt from earlier. This is an extra call to the model, so it adds cost and time.

The judge runs at `temperature=0`, which makes it prefer its most likely answer, so its output is more stable. Low temperature reduces variety, but it does not guarantee identical output.

Language models can miscount, so treat this answer as a second opinion. In the example run the model agreed with the Python vote. One agreement does not prove the model counts reliably, so compare the two answers each time you run the notebook.


In [31]:
# Temperature 0 so the judge gives the same answer on every run.
llm_judge = ChatOpenRouter(model=MODEL, temperature=0)

final_prompt = sc_prompt.format(query=query, responses=str(responses))

print(final_prompt)

Based on the responses (delimited by < >) to the following query, (delimited by triple backticks) return the response that occurs most frequently.

Query: ```There were nine computers in the server room. Five more computers were installed each day, from
monday to thursday. How many computers are now in the server room?
```

Responses: <['29', '29', '29', '29', '29']>



In [32]:
print(llm_judge.invoke(final_prompt).content)

29


### Limitations and responsible use

- **Cost and speed:** Five sampling calls cost roughly five times as much as one, and take longer to run. The optional judge adds one more call.
- **Works best with short, comparable answers:** Voting needs answers that can be matched exactly, such as a number or a single letter. For open-ended answers, such as a paragraph of advice, "most common" is hard to define.
- **Consistent is not always correct:** The model can repeat the same wrong reasoning five times. Agreement shows the model is stable, not that its answer is right.
- **Fragile answer extraction:** The code looks for a `Response:` line. If a sample has none, the code prints a message and skips it, so the vote can be based on fewer than five answers. Check the printed output.
- **The judge is also a model:** The LLM judge can make mistakes too. The Python vote is the more dependable of the two, because it is exact counting.
- **Results depend on the model:** Changing `MODEL` (or `SELF_CONSISTENCY_MODEL` in your `.env` file) can change the answers and how often they agree.
- **Check the data:** The examples come from a Hugging Face dataset. Check its licence and terms of use before you reuse them.
- **Not for high-stakes decisions:** Before relying on AI answers for anything important, such as medical, legal or financial advice, check them with a human expert.
